# Curación reproducible de código C++ de CodeNet

**Mesias Mariscal · Denise Rea**  
GEST. PROYECTOS DE SOFTWARE · NRC 36907 · ESPE  
**Iteración 001 · Preparación del futuro clasificador de posible IA**

El pipeline selecciona muestras históricas, conserva su procedencia, comprueba integridad y reserva ejercicios completos para evaluación. La ejecución no compila código de terceros, no entrena modelos y no necesita GPU.

La etiqueta `HISTORICO_CODENET` expresa el origen de la muestra. **No certifica autoría humana individual.**

## Recorrido
1. Configuración y entorno.
2. Fuente de datos y modos de ejecución.
3. Selección y exportación.
4. Distribución y controles de aceptación.
5. Reproducción independiente y pruebas.
6. Evidencia, límites y continuidad.

## 1 · Configuración y entorno

El modo predeterminado reproduce las 50 muestras incluidas. Para ampliar el corpus se editan los ejercicios y la cuota en `config/iteration1.json`. La configuración y los datos, junto con la versión del pipeline, determinan el resultado.

In [2]:
from pathlib import Path
import csv, json, os, sys, tempfile, time, subprocess, platform
from collections import Counter
from datetime import datetime, timezone
from xml.etree import ElementTree
from IPython.display import display, HTML, Markdown

ROOT = Path.cwd().resolve()
while not (ROOT / "config/iteration1.json").is_file() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if not (ROOT / "config/iteration1.json").is_file():
    raise RuntimeError("Abre el notebook dentro de la carpeta taller-sdd.")
sys.path.insert(0, str(ROOT / "src"))
from codenet_pipeline import run_pipeline, verify_manifest, extract_selected, dump_json, digest, VERSION

CONFIG = json.loads((ROOT / "config/iteration1.json").read_text(encoding="utf-8"))
EVIDENCE = ROOT / "evidence"
EVIDENCE.mkdir(exist_ok=True)
print("Python:", platform.python_version(), "| Pipeline:", VERSION)
print("Muestras por ejercicio:", CONFIG["samples_per_problem"], "| Ejercicios:", len(CONFIG["problems"]))
print("Semilla:", CONFIG["seed"], "| Corte UTC:", CONFIG["cutoff_utc"])
print("GPU requerida: no")

Python: 3.13.14 | Pipeline: 1.0.0
Muestras por ejercicio: 10 | Ejercicios: 5
Semilla: 20261021 | Corte UTC: 2021-05-01T00:00:00Z
GPU requerida: no


## 2 · Fuente de datos

**Modo ligero:** utiliza la fuente reducida distribuida con el proyecto. Reproduce el manifiesto y sus controles; no repite la exploración de todos los candidatos.

**Modo completo:** define `CODENET_ROOT` con la raíz expandida de CodeNet. También puedes definir `CODENET_ARCHIVE` y activar `REBUILD_FROM_ARCHIVE`; la lectura secuencial del comprimido puede tardar varios minutos. El destino de extracción debe estar vacío para evitar mezclar fuentes.

El registro `evidence/full_source_selection.json` conserva los conteos de la selección original sobre los metadatos completos de los cinco problemas.

In [3]:
REBUILD_FROM_ARCHIVE = False
archive = Path(os.environ.get("CODENET_ARCHIVE", str(Path.home() / "Downloads/Project_CodeNet.tar.gz")))
expanded_source = os.environ.get("CODENET_ROOT")
if REBUILD_FROM_ARCHIVE:
    SOURCE = extract_selected(archive, ROOT / "data/archive_cache", CONFIG["problems"])
    SOURCE_MODE = "archivo completo"
elif expanded_source:
    SOURCE = Path(expanded_source)
    SOURCE_MODE = "fuente expandida"
else:
    SOURCE = ROOT / "data/source_subset/Project_CodeNet"
    SOURCE_MODE = "subconjunto reproducible"
if not SOURCE.is_dir():
    raise FileNotFoundError(f"Fuente no disponible: {SOURCE}")
print("Modo:", SOURCE_MODE)
baseline_path = EVIDENCE / "full_source_selection.json"
baseline = json.loads(baseline_path.read_text(encoding="utf-8")) if baseline_path.exists() else None
if baseline:
    s = baseline["full_selection_summary"]
    print("Selección original: registros de metadatos =", s["metadata_rows"],
          "| elegibles por metadatos =", s["eligible_rows"], "| muestras =", s["selected"])

Modo: subconjunto reproducible
Selección original: registros de metadatos = 75699 | elegibles por metadatos = 13182 | muestras = 50


## 3 · Selección y exportación

La elegibilidad exige C++, estado `Accepted` y fecha anterior al corte. El orden estable depende de la semilla y del identificador. Cada ejercicio aporta usuarios distintos; las huellas normalizadas impiden repetir contenido idéntico salvo finales de línea. Las muestras exportadas conservan sus bytes originales.

El estado `Accepted` representa la aceptación del juez de programación. La primera iteración no vuelve a comprobar la corrección funcional.

In [4]:
started = time.perf_counter()
rows, summary, verification = run_pipeline(SOURCE, ROOT / "data/curated", CONFIG)
print("Muestras seleccionadas:", summary["selected"])
print("Tiempo del pipeline en este modo:", summary["duration_seconds"], "s")
print("Manifiesto SHA-256:", summary["manifest_sha256"])
print("Rechazos durante este recorrido:", summary["rejected"])
print("Entrenamiento de clasificación:", summary["classification_training_performed"])

Muestras seleccionadas: 50
Tiempo del pipeline en este modo: 1.0921 s
Manifiesto SHA-256: 108e436464cefd502a7d685f7e62f247818f8d9a26d9acc0e2188b9e33623610
Rechazos durante este recorrido: {}
Entrenamiento de clasificación: False


## 4 · Distribución y aceptación

Los tres ejercicios de entrenamiento aportan treinta archivos; validación y prueba aportan diez cada uno. El reparto inicial demuestra la separación por ejercicio y deberá ampliarse antes del experimento final de la tesis.

In [5]:
import html
def html_table(headers, records):
    head = "".join(f"<th>{html.escape(str(h))}</th>" for h in headers)
    body = "".join("<tr>" + "".join(f"<td>{html.escape(str(v))}</td>" for v in row) + "</tr>" for row in records)
    display(HTML("<style>table.codenet{border-collapse:collapse;width:100%;font-family:Arial,sans-serif}"
                 "table.codenet th{background:#0e7045;color:white;text-align:left}"
                 "table.codenet td,table.codenet th{padding:10px;border:1px solid #d9e1dc}"
                 "table.codenet tr:nth-child(even){background:#f2f6f3}</style>"
                 f"<table class='codenet'><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>"))
html_table(["Ejercicio", "Enunciado", "Tema", "Partición", "Muestras"],
    [(pid, info["title"], info["topic"], info["split"], summary["by_problem"][pid])
     for pid, info in CONFIG["problems"].items()])
html_table(["Control", "Resultado"], [(key, "CUMPLE" if value else "FALLA") for key, value in verification["checks"].items()])
print("Usuarios compartidos entre particiones:", verification["user_overlap"])
if not verification["all_passed"]:
    raise RuntimeError("Existen controles incumplidos; revisa verification.json.")

Ejercicio,Enunciado,Tema,Partición,Muestras
p02401,Simple Calculator,Operadores y selección,train,10
p02402,"Min, Max and Sum",Recorrido y acumulación,train,10
p02403,Print a Rectangle,Ciclos anidados,train,10
p02407,Reversing Numbers,Arreglos y recorrido inverso,validation,10
p02416,Sum of Numbers,Cadenas y suma de dígitos,test,10


Control,Resultado
quota,CUMPLE
required_fields,CUMPLE
eligibility,CUMPLE
unique_submission_ids,CUMPLE
unique_normalized_hashes,CUMPLE
unique_users_per_problem,CUMPLE
partition_assignment,CUMPLE
no_shared_problems,CUMPLE
integrity,CUMPLE
historical_label,CUMPLE


Usuarios compartidos entre particiones: {'train/validation': 0, 'train/test': 0, 'validation/test': 0}


### Trazabilidad de una muestra

El manifiesto conserva los identificadores anónimos, la fecha original, las rutas y las huellas. Los comentarios y nombres de variables no se eliminan: podrían aportar señales al futuro clasificador.

In [6]:
example = rows[0]
print(json.dumps({k: example[k] for k in ["submission_id", "problem_id", "user_id", "date_utc", "source_path", "sha256", "label", "label_basis"]}, ensure_ascii=False, indent=2))
print("\nFragmento de la muestra, mostrado como texto sin ejecución:\n")
print((ROOT / "data/curated" / example["curated_path"]).read_text(encoding="utf-8-sig")[:700])

{
  "submission_id": "s052030440",
  "problem_id": "p02401",
  "user_id": "u522482726",
  "date_utc": "2016-12-03T01:54:45Z",
  "source_path": "data/p02401/C++/s052030440.cpp",
  "sha256": "d7653d7aaee10bbcd380e8c2024f8c261d31c590a86159e9875fec31b51f6747",
  "label": "HISTORICO_CODENET",
  "label_basis": "Procedencia histórica de CodeNet; autoría individual no verificada"
}

Fragmento de la muestra, mostrado como texto sin ejecución:

#include <iostream>

using namespace std;

int main()
{
	int a = 0, b = 0;
	char op = 0;

	while(true)
	{
		cin >> a >> op >> b;
		if(op == '?'){ break; }
	
		switch (op)
		{
		case '+':
			cout << a + b << endl;
			break;
		case '-':
			cout << a - b << endl;
			break;
		case '*':
			cout << a * b << endl;
			break;
		case '/':
			cout << a / b << endl;
			break;
		}
	}


	return 0;
}


## 5 · Reproducción independiente y pruebas

La segunda selección utiliza un destino temporal independiente. El manifiesto debe coincidir byte a byte. Las pruebas incluyen metadatos inválidos, filtros, cuota, duplicados, corrupción, rutas y miembros inseguros del comprimido.

In [7]:
with tempfile.TemporaryDirectory(prefix="codenet-repro-") as temporary:
    _, repeat_summary, repeat_verification = run_pipeline(SOURCE, Path(temporary), CONFIG)
    same_manifest = (Path(temporary) / "manifest.csv").read_bytes() == (ROOT / "data/curated/manifest.csv").read_bytes()
if not same_manifest:
    raise RuntimeError("La segunda selección no reproduce el manifiesto.")
same_as_full_source = None
if baseline and summary["config_sha256"] == baseline["full_selection_summary"]["config_sha256"]:
    same_as_full_source = summary["manifest_sha256"] == baseline["full_selection_summary"]["manifest_sha256"]
    if not same_as_full_source:
        raise RuntimeError("El modo reducido no coincide con la selección original.")
print("Reproducción independiente:", same_manifest)
print("Coincidencia con selección original:", same_as_full_source)

result = subprocess.run([sys.executable, "-m", "pytest", "tests", "-q", "--junitxml=evidence/pytest.xml"], cwd=ROOT,
                        text=True, capture_output=True, encoding="utf-8", errors="replace")
print(result.stdout)
if result.returncode:
    print(result.stderr)
    raise RuntimeError("Las pruebas de aceptación no pasan.")
xml = ElementTree.parse(EVIDENCE / "pytest.xml").getroot()
suites = list(xml.iter("testsuite"))
test_stats = {key: sum(int(s.attrib.get(key, 0)) for s in suites) for key in ["tests", "failures", "errors", "skipped"]}

Reproducción independiente: True
Coincidencia con selección original: True
................                                                         [100%]
16 passed in 1.69s



## 6 · Evidencia y continuidad

Los controles acreditan la preparación del subconjunto. No representan precisión, recall o F1 del indicador IA: todavía no existe un clasificador entrenado en esta iteración.

Antes de entrenar: ampliar a veinte ejercicios, incorporar muestras IA bajo los mismos enunciados, revisar casi duplicados y problemas equivalentes, y cerrar las particiones del corpus final. La futura página docente consumirá un componente evaluado, conservando el indicador separado de la similitud entre entregas.

In [8]:
acceptance = {
    "iteration": "001-curacion-codenet", "source_mode": SOURCE_MODE,
    "selected": summary["selected"], "by_problem": summary["by_problem"], "by_split": summary["by_split"],
    "checks": verification["checks"], "all_checks_passed": verification["all_passed"],
    "user_overlap": verification["user_overlap"], "independent_reproduction": same_manifest,
    "matches_full_source_manifest": same_as_full_source, "manifest_sha256": summary["manifest_sha256"],
    "tests": test_stats, "python_version": platform.python_version(), "pipeline_version": VERSION,
    "classification_training_performed": False, "notebook_cells_completed": True,
    "recorded_at_utc": datetime.now(timezone.utc).isoformat(),
}
dump_json(EVIDENCE / "acceptance.json", acceptance)
display(Markdown(f"**Cierre de aceptación:** {summary['selected']} muestras, {len(verification['checks'])} controles correctos y {test_stats['tests']} pruebas sin fallos."))
print("Evidencia disponible en evidence/acceptance.json")

**Cierre de aceptación:** 50 muestras, 10 controles correctos y 16 pruebas sin fallos.

Evidencia disponible en evidence/acceptance.json


## Referencias

- [IBM Project CodeNet: datos, metadatos y controles de calidad](https://github.com/IBM/Project_CodeNet).
- [Project CodeNet: artículo del conjunto de datos](https://arxiv.org/abs/2105.12655).
- [CDLA Permissive 2.0](https://cdla.dev/permissive-2-0/). El acuerdo acompaña las muestras distribuidas.

La comprobación de huellas detecta igualdad exacta tras normalizar finales de línea; no detecta por sí sola copias con renombrado, reordenamiento o cambios algorítmicos.